# RETO 1: CLASIFICACION DE FLORES

### Importar librerías

In [ ]:
from urllib.request import urlretrieve
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

### Traer y leer Dataset

In [ ]:
irisURL = 'http://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'
urlretrieve(irisURL)
df_iris = pd.read_csv(irisURL, sep=',', names=["Largo_Sepalo", "Ancho_Sepalo", "Largo_Petalo", "Ancho_Petalo", "Clase"])

In [ ]:
print(df_iris)
print(df_iris.head())
print(df_iris.tail())

## <span style="color:yellow">Regresión Logística</span>

### Separar X, y, entrenamiento y pruebas

In [ ]:
X_log = df_iris.drop('Clase', axis=1).copy()
y_log = df_iris['Clase'].copy()

# print(X_log.head())
# print(y_log.head())

In [ ]:
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(X_log, y_log, test_size=0.2, random_state=42)

### Entrenar modelo logístico, predicciones

In [ ]:
model_logistic = LogisticRegression(max_iter=500)
model_logistic.fit(X_train_log, y_train_log)

In [ ]:
y_pred_logistic = model_logistic.predict(X_test_log)

# print(y_pred_logistic)

### Validaciones

In [ ]:
accuracy_log = accuracy_score(y_test_log, y_pred_logistic)

print(f"Precision Logistica: {accuracy_log}")

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test_log, y_pred_logistic)

## <span style="color:yellow">Regresión Lineal</span>

### Separar X, y, entrenamiento y pruebas

In [ ]:
X_lin = df_iris.drop('Clase', axis=1).copy()
le = LabelEncoder()
y_lin = le.fit_transform(df_iris['Clase']).copy()

In [ ]:
X_train_lin, X_test_lin, y_train_lin, y_test_lin = train_test_split(X_lin, y_lin, test_size=0.2, random_state=42)

### Entrenar modelo lineal, predicciones

In [ ]:
model_lineal = LinearRegression()
model_lineal.fit(X_train_lin, y_train_lin)

In [ ]:
y_pred_lineal = model_lineal.predict(X_test_lin)

# print(y_pred_lineal)

In [ ]:
y_rounded = np.round(y_pred_lineal)
y_rounded = np.clip(y_rounded, 0, 2)

print(y_rounded)

### Validaciones

In [ ]:
accuracy_lin = accuracy_score(y_test_lin, y_rounded)
mae = mean_absolute_error(y_test_lin, y_rounded)
mse = mean_squared_error(y_test_lin, y_rounded)

print(f"Precision Lineal: {accuracy_lin}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test_lin, y_rounded)

## <span style="color:yellow">Arboles de Decisión</span>

### Separar X, y, entrenamiento y pruebas

In [ ]:
X_tree = df_iris.drop('Clase', axis=1).copy()
y_tree = df_iris['Clase'].copy()

In [ ]:
X_train_tree, X_test_tree, y_train_tree, y_test_tree = train_test_split(X_tree, y_tree, test_size=0.2, random_state=42)

### Entrenar modelo arbol de decisión, predicciones

In [ ]:
tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train_tree, y_train_tree)

print("Profundidad del árbol sin Poda:", tree.get_depth())
print("Número de hojas sin Poda:", tree.get_n_leaves())

In [ ]:
path = tree.cost_complexity_pruning_path(X_train_tree, y_train_tree)
ccp_alphas = path.ccp_alphas

print("Valores de ccp_alpha:", ccp_alphas)

In [ ]:
trees = []
for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=alpha)
    clf.fit(X_train_tree, y_train_tree)
    trees.append(clf)

In [ ]:
train_scores = []
test_scores = []

for clf in trees:
    train_scores.append(clf.score(X_train_tree, y_train_tree))
    test_scores.append(clf.score(X_test_tree, y_test_tree))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(ccp_alphas, train_scores, label="Entrenamiento", marker='o')
plt.plot(ccp_alphas, test_scores, label="Prueba", marker='o')
plt.xlabel("Alpha")
plt.ylabel("Precisión")
plt.title("Precision vs Alpha para Árbol de Decisión")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_index = np.argmax(test_scores)
best_alpha = ccp_alphas[best_index]
print("Mejor alpha:", best_alpha)
print("Precision con mejor alpha:", test_scores[best_index])

In [ ]:
final_tree = DecisionTreeClassifier(criterion='entropy', random_state=42, ccp_alpha=best_alpha)
final_tree.fit(X_train_tree, y_train_tree)
print("Profundidad del árbol final:", final_tree.get_depth())
print("Número de hojas del árbol final:", final_tree.get_n_leaves())

In [ ]:
y_pred_tree = tree.predict(X_test_tree)

print(y_pred_tree)

### Validaciones

In [ ]:
accuracy_tree = accuracy_score(y_test_tree, y_pred_tree)

print(f"Precision Arbol de Decision: {accuracy_tree}")

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test_tree, y_pred_tree)

In [ ]:
plt.figure(figsize=(12,8))
plot_tree(final_tree, filled=True, feature_names=X_tree.columns, class_names=tree.classes_.astype(str))
plt.show()

## <span style="color:yellow">Bosques Aleatorios</span>

### Separar X, y, entrenamiento y pruebas

In [ ]:
X_forest = df_iris.drop('Clase', axis=1).copy()
y_forest = df_iris['Clase'].copy()

In [ ]:
X_train_forest, X_test_forest, y_train_forest, y_test_forest = train_test_split(X_forest, y_forest, test_size=0.2, random_state=42)

### Entrenar modelo arbol de decisión, predicciones

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42, max_features=2)
rf.fit(X_train_forest, y_train_forest)

In [ ]:
y_pred_forest = rf.predict(X_test_forest)

print(y_pred_forest)

### Validaciones

In [ ]:
accuracy_forest = accuracy_score(y_test_forest, y_pred_forest)

print(f"Precision Bosque Aleatorio: {accuracy_forest}")

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test_forest, y_pred_forest)

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X_forest.columns)
importancias.sort_values().plot(kind='barh')
plt.title("Importancia de las características en el Bosque Aleatorio")
plt.xlabel("Importancia")

## <span style="color:red">Predecir</span>

In [ ]:
nueva_flor = np.array([[8, 3, 8, 0.2]])

prediccion_logistica = model_logistic.predict(nueva_flor)
prediccion_lineal = model_lineal.predict(nueva_flor)
prediccion_final_tree = final_tree.predict(nueva_flor)
prediccion_rf = rf.predict(nueva_flor)

print(f"Predicción Logística: {prediccion_logistica[0]}")
print(f"Predicción Lineal: {le.inverse_transform([int(round(prediccion_lineal[0]))])[0]}")
print(f"Predicción Árbol de Decisión: {prediccion_final_tree[0]}")
print(f"Predicción Bosque Aleatorio: {prediccion_rf[0]}")

## <span style="color:red">Conclusiones</span>

<p>De lo obsevado podemos observar lo siguiente:</p>

<ul>
    <li>
        La <strong>Regresión Lineal</strong> no es un buen modelo para clasificar en general, y mucho menos el dataset Iris que tiene como salida una clase y no un dato numérico. Por lo que para que "funcione" se debe realizar una conversión previa de las clases a una interpretación numérica, y además cuando el modelo se entrena no va predecir números enteros sino confusos que incluso pueden ser negativos, resultando en que para hacer la predicción de clase de la flor, se debe redondear y esto es una mala práctica que puede resultar en predicciones erroneas. En este caso "funcionó" porque el dataset Iris es pequeño y muy separable.
    </li>
    <li>
        Cuando utilizamos el metodo <strong>Arból de Decision</strong> encontramos que no genera fronteras lineales sino que se divide el espacio mediante reglas binarias, además que es un método que se debe saber controlar, porque si no tiende al sobreajuste, esto lo hicimos principalmente con Poda, obteniendo unos alphas para penalizar complejidad y ver las diferentes precisiones, y elegir el mejor valor para obtener el mejor arbol. Además, también se puede restringir la cantidad de niveles y/o hojas que tendrá el arbol. El árbol tiene alta varianza si no se controla.
    </li>
    <li>
        Para el método <strong>Bosques Aleatorios</strong> pudimos ver que es una versión mejorada respecto a un solo árbol, pues es menos sensible al sobreajuste y generaliza mejor, además que permite ver la importancia de las variables. Pudimos modificar la cantidad de arboles que se generan y la profundización de los mismos. Y por ello creemos que es el mejor método para clasificar, tiene bastantes parámetros a modificar y además hace las iteraciones suficientes para llegar a un buen modelo. 
    </li>
    <li>
        Comparando entre diferentes modelos, la mayoría tienen bajo riesgo de sobreajuste exceptuando el <strong>Arbol de Decision</strong> si no se controla. La interpretabilidad que tiene la <strong>Regresión Lineal</strong> es baja, mientras que la de la <strong>Regresión Logistica</strong> y <strong>Arbol de Decisión</strong> es mayor. Y además, concluimos que el rendimiento para el dataset Iris es aceptable para <strong>Regresión Lineal</strong> solo porque funciona por coincidencia, mientras que los demás si son excelentes modelos, tanto así que la precisión que se alcanza es siempre 1 o perfecta.
    </li>
    <li>
        En todos los modelos aplicados, obtuvimos una precision de 1, es decir, que la capacidad de los modelos de predecir era perfecta, esto sucede principalmente porque el dataset usado es bastante pequeño para aprendizaje de máquina y además tiene datos bastantes separables, entonces tampoco se necesita el modelo más potente para hacer una buena predición sobre el mismo. Realmente no logramos ver una diferencia en resultados ni en las precisiones ni en las matrices de confusión, solamente en la capacidad que tienen los modelos de procesar y de ser modificados sus parámetros.
    </li>
</ul>